<a href="https://colab.research.google.com/github/isabellapolido-coder/Calculo-Num-Lista-1/blob/main/CN_Lista_4_Q4_Q5_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cálculo Numérico — Projeto 4: Solução Numérica de EDOs
**Questões 4 e 5**

Neste notebook resolvemos numericamente dois problemas de valor inicial (PVI) usando quatro métodos de passo simples: **Euler, Heun, Ponto Médio e Runge-Kutta de ordem 4 (RK4)**.

**Organização**
0. Preparação: bibliotecas e implementação dos métodos
1. Questão 4 — PVI $y' = y - t^2 + 1$
2. Questão 5 — Lei de resfriamento de Newton

Todas as células podem ser executadas em ordem (*Ambiente de execução → Executar tudo*).

## 0. Preparação

### 0.1 Bibliotecas
- `numpy`: vetores e funções matemáticas (`exp`, `log`).
- `matplotlib`: gráficos.
- `pandas`: tabelas de resultados.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 110

### 0.2 Os métodos numéricos

Todos resolvem o PVI
$$y' = f(t, y), \qquad y(t_0) = y_0$$
dividindo $[t_0, t_f]$ em $n = (t_f - t_0)/h$ passos de tamanho $h$, com $t_i = t_0 + ih$.

| Método | Fórmula | Ordem | Avaliações de $f$ por passo |
|---|---|---|---|
| **Euler** | $y_{i+1} = y_i + h\,f(t_i, y_i)$ | 1 | 1 |
| **Heun** | $k_1 = f(t_i,y_i)$, $k_2 = f(t_i+h,\ y_i+hk_1)$, $y_{i+1} = y_i + \frac{h}{2}(k_1+k_2)$ | 2 | 2 |
| **Ponto Médio** | $k_1 = f(t_i,y_i)$, $k_2 = f(t_i+\frac h2,\ y_i+\frac h2 k_1)$, $y_{i+1} = y_i + h\,k_2$ | 2 | 2 |
| **RK4** | $y_{i+1} = y_i + \frac h6(k_1+2k_2+2k_3+k_4)$ | 4 | 4 |

**Ideia de cada um**
- **Euler** segue a reta tangente no início do passo (usa só a inclinação em $t_i$).
- **Heun** é um *preditor-corretor*: prevê $y_{i+1}$ com Euler e depois usa a **média** das inclinações nos dois extremos do passo.
- **Ponto Médio** usa a inclinação estimada no **meio** do passo.
- **RK4** combina quatro inclinações (início, duas no meio, fim) com pesos $1,2,2,1$ (regra de Simpson).

A *ordem* $p$ significa que o erro global é $O(h^p)$: dividir $h$ por 2 reduz o erro por um fator $\approx 2^p$.

In [2]:
def euler(f, t0, y0, tf, h):
    n = int(round((tf - t0) / h))          # número de passos
    t = t0 + h * np.arange(n + 1)          # malha de tempo
    y = np.empty(n + 1); y[0] = y0         # condição inicial
    for i in range(n):
        y[i + 1] = y[i] + h * f(t[i], y[i])
    return t, y


def heun(f, t0, y0, tf, h):
    n = int(round((tf - t0) / h))
    t = t0 + h * np.arange(n + 1)
    y = np.empty(n + 1); y[0] = y0
    for i in range(n):
        k1 = f(t[i], y[i])                 # inclinação no início
        k2 = f(t[i] + h, y[i] + h * k1)    # inclinação no fim (previsto por Euler)
        y[i + 1] = y[i] + h / 2 * (k1 + k2)
    return t, y


def ponto_medio(f, t0, y0, tf, h):
    n = int(round((tf - t0) / h))
    t = t0 + h * np.arange(n + 1)
    y = np.empty(n + 1); y[0] = y0
    for i in range(n):
        k1 = f(t[i], y[i])
        k2 = f(t[i] + h / 2, y[i] + h / 2 * k1)   # inclinação no ponto médio
        y[i + 1] = y[i] + h * k2
    return t, y


def rk4(f, t0, y0, tf, h):
    n = int(round((tf - t0) / h))
    t = t0 + h * np.arange(n + 1)
    y = np.empty(n + 1); y[0] = y0
    for i in range(n):
        k1 = f(t[i], y[i])
        k2 = f(t[i] + h / 2, y[i] + h / 2 * k1)
        k3 = f(t[i] + h / 2, y[i] + h / 2 * k2)
        k4 = f(t[i] + h, y[i] + h * k3)
        y[i + 1] = y[i] + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return t, y


METODOS = {"Euler": euler, "Heun": heun, "Ponto Médio": ponto_medio, "RK4": rk4}

### 0.3 Teste rápido dos métodos
Antes de aplicar nos problemas, testo com $y' = y$, $y(0)=1$ (solução $e^t$). Em $t=1$ o valor exato é $e = 2.718281828...$. Os métodos devem se aproximar dele na ordem Euler < Heun/Ponto Médio < RK4.

In [3]:
for nome, met in METODOS.items():
    _, y = met(lambda t, y: y, 0, 1, 1, 0.1)
    print(f"{nome:12s} y(1) ≈ {y[-1]:.8f}   erro = {abs(np.e - y[-1]):.2e}")
print(f"{'Exato':12s} y(1) = {np.e:.8f}")

Euler        y(1) ≈ 2.59374246   erro = 1.25e-01
Heun         y(1) ≈ 2.71408085   erro = 4.20e-03
Ponto Médio  y(1) ≈ 2.71408085   erro = 4.20e-03
RK4          y(1) ≈ 2.71827974   erro = 2.08e-06
Exato        y(1) = 2.71828183


---
# Questão 4

$$y'(t) = y(t) - t^2 + 1,\qquad 0 \le t \le 2,\qquad y(0) = 0.5$$

Solução analítica: $y(t) = (t+1)^2 - \tfrac12 e^t$.

**Verificação:** $y(0) = 1 - \tfrac12 = 0.5$ ✓. Derivando, $y' = 2(t+1) - \tfrac12 e^t$ e $y - t^2 + 1 = (t+1)^2 - \tfrac12e^t - t^2 + 1 = 2t + 2 - \tfrac12 e^t$ ✓.

### Itens 1 e 2 — Implementação e resolução com $h = 0.2,\ 0.1,\ 0.05$
Os métodos já foram implementados na seção 0. Aqui defino o problema e resolvo para cada método e cada passo, guardando tudo em um dicionário `sol4[(método, h)] = (t, y)`.

In [ ]:
# Definição do PVI
f4 = lambda t, y: y - t**2 + 1            # lado direito da EDO
y4 = lambda t: (t + 1)**2 - 0.5*np.exp(t) # solução analítica
t0, y0, tf = 0.0, 0.5, 2.0
passos4 = [0.2, 0.1, 0.05]

# Resolve para todos os métodos e passos
sol4 = {}
for nome, met in METODOS.items():
    for h in passos4:
        sol4[(nome, h)] = met(f4, t0, y0, tf, h)

print(f"Solução exata: y(2) = {y4(2.0):.10f}\n")
aprox4 = pd.DataFrame({m: [sol4[(m, h)][1][-1] for h in passos4] for m in METODOS},
                      index=[f"h = {h}" for h in passos4])
print("Valor aproximado y_h(2):")
aprox4

### Item 3 — Gráficos: solução numérica × solução analítica
Um painel por método, com os três passos. A curva preta é a solução exata.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 9))
tt = np.linspace(0, 2, 400)
for ax, nome in zip(axs.ravel(), METODOS):
    ax.plot(tt, y4(tt), "k-", lw=2, label="Exata")
    for h, mk in zip(passos4, ["o--", "s--", "^--"]):
        t, y = sol4[(nome, h)]
        ax.plot(t, y, mk, ms=3, lw=1, label=f"h = {h}")
    ax.set_title(f"Q4 — {nome}")
    ax.set_xlabel("t"); ax.set_ylabel("y(t)")
    ax.grid(alpha=.3); ax.legend()
plt.tight_layout()
plt.show()

**Leitura dos gráficos:** o Euler fica abaixo da solução exata e o erro cresce com $t$ (a solução cresce exponencialmente e o erro acumula). Heun, Ponto Médio e RK4 praticamente coincidem com a curva exata na escala do gráfico, então as diferenças só aparecem nos erros numéricos a seguir.

### Itens 4 e 5 — Erro absoluto em $t=2$ e tabela comparativa
$$E(h) = |y(2) - y_h(2)|$$

In [ ]:
erros4 = {m: [abs(y4(2.0) - sol4[(m, h)][1][-1]) for h in passos4] for m in METODOS}
tab4 = pd.DataFrame(erros4, index=[f"h = {h}" for h in passos4])
print("Erro absoluto E(h) em t = 2:")
tab4.style.format("{:.4e}")

### Item 6 — Ordem de convergência experimental
Se $E(h) \approx C\,h^p$, então $E(h)/E(h/2) \approx 2^p$, logo
$$p \approx \frac{\ln\big(E(h)/E(h/2)\big)}{\ln 2}.$$

In [ ]:
ordens4 = {}
for m in METODOS:
    e = erros4[m]
    ordens4[m] = [np.log(e[0]/e[1])/np.log(2),    # h = 0.2  -> 0.1
                  np.log(e[1]/e[2])/np.log(2)]    # h = 0.1  -> 0.05
ord4 = pd.DataFrame(ordens4, index=["0.2 → 0.1", "0.1 → 0.05"])
print("Ordem experimental p:")
ord4.style.format("{:.3f}")

In [ ]:
# Gráfico log-log: a inclinação de cada reta é a ordem do método
fig, ax = plt.subplots(figsize=(6.5, 5))
for m in METODOS:
    ax.loglog(passos4, erros4[m], "o-", label=m)
ax.set_xlabel("h"); ax.set_ylabel("E(h) em t = 2")
ax.set_title("Q4 — Convergência (log-log)")
ax.grid(alpha=.3, which="both"); ax.legend()
plt.tight_layout(); plt.show()

### Item 7 — Discussão

- **Resultado:** os valores de $p$ confirmam a teoria: Euler $\to 1$, Heun e Ponto Médio $\to 2$, RK4 $\to 4$. O $p$ do Euler (0.86 e depois 0.93) ainda se aproxima de 1 lentamente porque os passos são relativamente grandes.
- **Precisão para o mesmo $h$:** RK4 > Ponto Médio > Heun > Euler. Com $h=0.2$, o erro do RK4 ($1.1\times10^{-4}$) é cerca de 4000 vezes menor que o do Euler ($4.4\times10^{-1}$).
- **Custo × benefício:** o RK4 usa 4 avaliações de $f$ por passo contra 1 do Euler. Comparando com o **mesmo número de avaliações** (40): RK4 com $h=0.2$ (10 passos × 4) tem erro $1.1\times10^{-4}$, enquanto Euler com $h=0.05$ (40 passos × 1) tem erro $1.3\times10^{-1}$. O RK4 é cerca de 1000 vezes mais preciso.
- **Heun × Ponto Médio:** têm a mesma ordem; a diferença (o Ponto Médio teve erro ≈ 4.8 vezes menor) está na constante do erro, que depende da equação.
- **Efeito de reduzir $h$:** ao dividir $h$ por 2, o erro cai $\approx 2\times$ (Euler), $\approx 4\times$ (Heun e Ponto Médio) e $\approx 16\times$ (RK4).

---
# Questão 5 — Lei de resfriamento de Newton

$$\frac{dT}{dt} = -k\,(T - T_a),\qquad T(0)=90\,^\circ\text{C},\quad T_a = 25\,^\circ\text{C},\quad k = 0.15\ \text{min}^{-1}$$

### Item 1 — Solução analítica
A EDO é linear e separável. Seja $u = T - T_a$. Então $u' = T' = -ku$, cuja solução é $u(t) = u(0)\,e^{-kt}$. Com $u(0) = 90 - 25 = 65$:

$$\boxed{T(t) = 25 + 65\,e^{-0.15\,t}}$$

**Interpretação:** a temperatura decai exponencialmente em direção à temperatura ambiente $T_a = 25\,^\circ$C. A escala de tempo do processo é $1/k \approx 6.7$ min.

In [ ]:
k, Ta, T0 = 0.15, 25.0, 90.0
f5 = lambda t, T: -k * (T - Ta)               # lado direito da EDO
T5 = lambda t: Ta + (T0 - Ta)*np.exp(-k*t)    # solução analítica

print(f"T(0)  = {T5(0):.4f} °C")
print(f"T(30) = {T5(30):.8f} °C  (tende a Ta = 25 °C)")

### Itens 2 e 3 — Métodos numéricos em $0 \le t \le 30$ min com $h = 1,\ 0.5,\ 0.25$
Uso apenas Euler, Heun e RK4, como pede o enunciado.

In [ ]:
met5 = {m: METODOS[m] for m in ["Euler", "Heun", "RK4"]}
passos5 = [1.0, 0.5, 0.25]

sol5 = {}
for nome, met in met5.items():
    for h in passos5:
        sol5[(nome, h)] = met(f5, 0.0, T0, 30.0, h)

print("T_h(30) numérico:")
pd.DataFrame({m: [sol5[(m, h)][1][-1] for h in passos5] for m in met5},
             index=[f"h = {h}" for h in passos5]).style.format("{:.8f}")

In [ ]:
# Temperaturas em alguns instantes (h = 1)
ts = [0, 5, 10, 15, 20, 25, 30]
linhas = {"Exata": [T5(t) for t in ts]}
for m in met5:
    t, T = sol5[(m, 1.0)]
    linhas[m] = [T[int(round(x))] for x in ts]      # h = 1 -> índice = t
pd.DataFrame(linhas, index=[f"t = {x}" for x in ts]).style.format("{:.5f}")

### Item 4 — Gráfico: solução analítica × numéricas

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
tt = np.linspace(0, 30, 400)
for ax, h in zip(axs, [1.0, 0.5]):
    ax.plot(tt, T5(tt), "k-", lw=2, label="Analítica")
    for m, mk in zip(met5, ["o--", "s--", "^--"]):
        t, T = sol5[(m, h)]
        ax.plot(t, T, mk, ms=3, lw=1, label=m)
    ax.axhline(40, color="gray", ls=":", label="T = 40 °C")
    ax.set_title(f"Q5 — Resfriamento (h = {h})")
    ax.set_xlabel("t (min)"); ax.set_ylabel("T (°C)")
    ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

### Item 5 — Instante em que $T = 40\,^\circ$C

**Valor exato:** $40 = 25 + 65e^{-0.15t} \Rightarrow t = \dfrac{\ln(65/15)}{0.15} \approx 9.7756$ min.

**Valor numérico:** o primeiro ponto com $T_i \le 40$ fica logo depois do cruzamento. Interpolo **linearmente** entre esse ponto e o anterior:
$$t^* = t_{i-1} + \frac{40 - T_{i-1}}{T_i - T_{i-1}}\,(t_i - t_{i-1})$$

In [ ]:
t_exato = np.log((T0 - Ta)/(40 - Ta))/k
print(f"Instante exato: t = ln(65/15)/0.15 = {t_exato:.6f} min\n")

def tempo_alvo(t, T, alvo):
    i = np.where(T <= alvo)[0][0]                  # primeiro ponto abaixo do alvo
    return t[i-1] + (alvo - T[i-1])*(t[i] - t[i-1])/(T[i] - T[i-1])

res = {m: [tempo_alvo(*sol5[(m, h)], 40.0) for h in passos5] for m in met5}
print("Instante (min) em que T = 40 °C, por interpolação linear:")
pd.DataFrame(res, index=[f"h = {h}" for h in passos5]).style.format("{:.5f}")

**Conclusão:** $t \approx 9.78$ min (cerca de 9 min e 47 s). O RK4 já acerta com $h=1$; o Euler subestima bastante porque sua temperatura cai mais rápido que a real. O pequeno resíduo do RK4 (≈ 0.01 min) vem da **interpolação linear**, não do método.

### Item 6 — Erro absoluto em $t = 30$

In [ ]:
erros5 = {m: [abs(T5(30.0) - sol5[(m, h)][1][-1]) for h in passos5] for m in met5}
tab5 = pd.DataFrame(erros5, index=[f"h = {h}" for h in passos5])
print("Erro absoluto em t = 30:")
tab5.style.format("{:.4e}")

In [ ]:
# Gráfico do erro ao longo do tempo (h = 1)
fig, ax = plt.subplots(figsize=(7, 5))
for m in met5:
    t, T = sol5[(m, 1.0)]
    ax.semilogy(t, np.abs(T - T5(t)) + 1e-18, label=m)
ax.set_xlabel("t (min)"); ax.set_ylabel("|erro|")
ax.set_title("Q5 — Erro absoluto ao longo do tempo (h = 1)")
ax.grid(alpha=.3, which="both"); ax.legend()
plt.tight_layout(); plt.show()

### Item 7 — Influência do tamanho do passo

- Reduzir $h$ melhora todos os métodos, mas em taxas diferentes. Ao dividir $h$ por 2, o erro cai $\approx 2\times$ no Euler (ordem 1), $\approx 4\times$ no Heun (ordem 2) e $\approx 16\times$ no RK4 (ordem 4).
- Aqui a escala de tempo do problema é $1/k \approx 6.7$ min. Com $h \le 1$ há vários passos por constante de tempo, então todos os métodos são **estáveis** e o erro observado é só de **truncamento**.
- O Euler tem um erro sistemático: a temperatura numérica fica sempre **abaixo** da real, e esse erro se acumula ao longo do tempo.
- O RK4 com $h=1$ (erro $1.6\times10^{-5}$) é mais preciso que o Euler com $h=0.25$ (erro $6\times10^{-2}$), mesmo com passo 4 vezes maior.

---
## Resumo geral

| | Euler | Heun | Ponto Médio | RK4 |
|---|---|---|---|---|
| Ordem | 1 | 2 | 2 | 4 |
| Avaliações de $f$/passo | 1 | 2 | 2 | 4 |
| Ordem medida na Q4 | ≈ 0.9 | ≈ 2.0 | ≈ 2.0 | ≈ 4.0 |

O RK4 é o método mais eficiente nos dois problemas: para o mesmo esforço computacional, o erro é ordens de grandeza menor.